# Agri Yield Prediction — Pipeline Preprocessing & Ensemble
**Train: 16,000 rows | Test: 3,999 rows | 50 features**

Pipeline theo docx: BoxCox transform + Label Encoding + HistGradientBoostingRegressor (max_leaf_nodes=63)

In [1]:
# ── Cell 1: Import libraries ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
from scipy.stats import boxcox
from scipy.special import inv_boxcox
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded ✅')

Libraries loaded ✅


In [2]:
# ── Cell 2: Load data ─────────────────────────────────────────────────────────
train_df = pd.read_csv('Agri_Train_Combined_16k.csv')
test_df  = pd.read_csv('Agri_Test_Original_4k.csv')

print(f'Train shape : {train_df.shape}')
print(f'Test  shape : {test_df.shape}')
train_df.head(3)

Train shape : (16000, 51)
Test  shape : (3999, 51)


,Area,District,Season,Avg Temp,Avg Humidity,Crop Name,Transplant,Growth,Harvest,Production,...,Heat_Stress_Days,Wind_Mean,Wind_Max,Rain_Temp_Ratio,Extreme_Heat_Risk,Is_Extreme_Heat,LST_C,is_extreme_Heat_Stress_Days,is_extreme_Wind_Max,Yield
0,181.365458,Sherpur,Kharif 1,25.0,72.5,Kakrol,April,May To July,July To Sep,407.871878,...,34.73869,2.333177,6.48,45.39,High Risk,1,28.243465,0,0,2.2489
1,442.971087,Munshiganj,Kharif 2,16.0,85.0,Radish,October,Nov To Feb,March,2681.928388,...,0.00000,2.140872,7.53,41.18,Low Risk,0,28.001091,0,0,6.0544
2,734.868770,Chuadanga,Rabi,20.0,75.0,Lal Shak,September,Oct To Nov,Nov To Feb,1476.223863,...,0.00000,1.522141,4.59,4.55,Low Risk,0,25.629617,0,0,2.0088


In [3]:
# ── Cell 3: Step 1 — Categorical Encoding (Label Encode) ─────────────────────
# Thực hiện TRƯỚC khi tách X/y
# Target Encoding đã được thử và cho kết quả THẤP HƠN (delta = −0.003)
# HistGB học grouping tốt từ label encoding → dùng Label Encode

TARGET = 'Yield'

cat_cols = [
    'District', 'Season', 'Crop Name', 'Transplant', 'Growth',
    'Harvest', 'Dominant_Soil_Texture', 'Water_Availability_Cat',
    'pH_Suitability', 'Extreme_Heat_Risk', 'Is_Extreme_Heat'
]

label_encoders = {}

for c in cat_cols:
    if c in train_df.columns:
        le = LabelEncoder()
        # fit trên union train+test để tránh unseen label error
        all_vals = pd.concat([train_df[c], test_df[c]]).astype(str).unique()
        le.fit(all_vals)
        train_df[c] = le.transform(train_df[c].astype(str))
        test_df[c]  = le.transform(test_df[c].astype(str))
        label_encoders[c] = le

print(f'Encoded {len(cat_cols)} categorical columns ✅')
print(f'Columns encoded: {cat_cols}')
print(f'\nSample train_df sau encoding (5 dòng đầu):')
train_df[cat_cols].head()

Encoded 11 categorical columns ✅
Columns encoded: ['District', 'Season', 'Crop Name', 'Transplant', 'Growth', 'Harvest', 'Dominant_Soil_Texture', 'Water_Availability_Cat', 'pH_Suitability', 'Extreme_Heat_Risk', 'Is_Extreme_Heat']

Sample train_df sau encoding (5 dòng đầu):


,District,Season,Crop Name,Transplant,Growth,Harvest,Dominant_Soil_Texture,Water_Availability_Cat,pH_Suitability,Extreme_Heat_Risk,Is_Extreme_Heat
0,58,0,36,0,18,16,1,1,1,0,1
1,38,1,60,10,22,22,1,1,1,1,0
2,10,2,40,11,27,30,1,1,1,1,0
3,60,2,37,3,16,4,1,1,1,1,0
4,15,1,2,1,30,27,1,1,1,1,0


In [4]:
# ── Cell 4: Step 2 — Target Transform: BoxCox (+0.040 R²) ────────────────────
# Thực hiện TRƯỚC khi tách X/y
# Lớn nhất trong toàn pipeline: skewness 3.53 → 0.0, lambda ≈ 0.046 (≈ log)

y_raw = train_df[TARGET].copy()
y_bc_arr, lambda_ = boxcox(y_raw + 1e-6)   # +1e-6 tránh log(0)

# Ghi đè cột Yield trong train_df bằng giá trị đã transform
train_df[TARGET] = y_bc_arr

print(f'BoxCox lambda  : {lambda_:.4f}  (≈ 0 = log transform)')
print(f'Skewness trước : {y_raw.skew():.3f}')
print(f'Skewness sau   : {pd.Series(y_bc_arr).skew():.3f}')
print(f'\nYield range sau BoxCox — min={train_df[TARGET].min():.3f}, max={train_df[TARGET].max():.3f}')
print(f'\nSample train_df[Yield] sau BoxCox (5 dòng đầu):')
train_df[[TARGET]].head()

BoxCox lambda  : 0.0459  (≈ 0 = log transform)
Skewness trước : 3.528
Skewness sau   : -0.000

Yield range sau BoxCox — min=-2.644, max=4.609

Sample train_df[Yield] sau BoxCox (5 dòng đầu):


,Yield
0,0.825697
1,1.877267
2,0.708820
3,0.668980
4,-1.494623


In [5]:
# ── Cell 4b: Export data đã xử lý (sau Encoding + BoxCox, trước khi tách X/y) ─
train_df.to_csv('train_processed.csv', index=False)
test_df.to_csv('test_processed.csv',  index=False)

print('Exported data đã xử lý:')
print('  train_processed.csv  — categorical encoded + Yield BoxCox-transformed')
print('  test_processed.csv   — categorical encoded, Yield giữ nguyên (raw)')
print(f'\ntrain_processed shape : {train_df.shape}')
print(f'test_processed  shape : {test_df.shape}')

Exported data đã xử lý:
  train_processed.csv  — categorical encoded + Yield BoxCox-transformed
  test_processed.csv   — categorical encoded, Yield giữ nguyên (raw)

train_processed shape : (16000, 51)
test_processed  shape : (3999, 51)


In [6]:
# ── Cell 5: Tách X / y (biến mục tiêu = Yield) ───────────────────────────────
X_train = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].copy()          # BoxCox-transformed

X_test  = test_df.drop(columns=[TARGET]).copy()
y_test  = test_df[TARGET].copy()           # Yield gốc (ton/ha)

print(f'X_train : {X_train.shape}  |  y_train : {y_train.shape}  (BoxCox-transformed)')
print(f'X_test  : {X_test.shape}   |  y_test  : {y_test.shape}   (raw ton/ha)')
print(f'\ny_train — min={y_train.min():.3f}, max={y_train.max():.3f}, mean={y_train.mean():.3f}')
print(f'y_test  — min={y_test.min():.3f}, max={y_test.max():.3f}, mean={y_test.mean():.3f}')

# ── Export CSV ────────────────────────────────────────────────────────────────
X_train.to_csv('X_train.csv', index=False)
X_test.to_csv('X_test.csv',  index=False)
y_train.to_csv('y_train.csv', index=False, header=['Yield_BoxCox'])
y_test.to_csv('y_test.csv',  index=False, header=['Yield'])

print('\nExported:')
print('  X_train.csv  — features train (encoded, no Yield)')
print('  X_test.csv   — features test  (encoded, no Yield)')
print('  y_train.csv  — target train   (BoxCox-transformed)')
print('  y_test.csv   — target test    (raw ton/ha)')

X_train : (16000, 50)  |  y_train : (16000,)  (BoxCox-transformed)
X_test  : (3999, 50)   |  y_test  : (3999,)   (raw ton/ha)

y_train — min=-2.644, max=4.609, mean=0.930
y_test  — min=0.014, max=98.991, mean=4.180

Exported:
  X_train.csv  — features train (encoded, no Yield)
  X_test.csv   — features test  (encoded, no Yield)
  y_train.csv  — target train   (BoxCox-transformed)
  y_test.csv   — target test    (raw ton/ha)


In [7]:
# ── Cell 6: Step 3 — Train Model: HistGradientBoostingRegressor ──────────────
# max_leaf_nodes=63 (default 31 → 63): MAE_high giảm 11.5%, free gain
# Feature Engineering không cần: HistGB học implicit interactions qua tree splits

model = HistGradientBoostingRegressor(
    max_iter       = 300,
    max_leaf_nodes = 63,   # key change: default 31 → 63
    random_state   = 42
)

model.fit(X_train.fillna(0), y_train)
print('Model trained ✅')

Model trained ✅


In [8]:
# ── Cell 7: Step 4 — Predict & Inverse BoxCox ────────────────────────────────

y_pred_bc = model.predict(X_test.fillna(0))
y_pred    = inv_boxcox(y_pred_bc, lambda_)   # đưa về đơn vị gốc (ton/ha)

print(f'Prediction range: [{y_pred.min():.3f}, {y_pred.max():.3f}] ton/ha')

Prediction range: [0.097, 58.685] ton/ha


In [9]:
# ── Cell 8: Đánh giá kết quả toàn bộ test set ────────────────────────────────

test_r2  = r2_score(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)

print('=' * 45)
print(f'  Test R²  : {test_r2:.4f}   (target ≥ 0.90)')
print(f'  Test MAE : {test_mae:.3f} ton/ha')
print('=' * 45)

# R² theo từng yield range
ranges = [(0, 2), (2, 5), (5, 10), (10, 20), (20, 9999)]
labels = ['0-2', '2-5', '5-10', '10-20', '20+']

print('\nR² theo Yield Range:')
print(f'  {"Range":>8}  {"R²":>8}  {"MAE":>8}  {"N samples":>10}')
print('  ' + '-' * 44)
for (lo, hi), lbl in zip(ranges, labels):
    mask = (y_test >= lo) & (y_test < hi)
    if mask.sum() < 2:
        continue
    r2  = r2_score(y_test[mask], y_pred[mask])
    mae = mean_absolute_error(y_test[mask], y_pred[mask])
    print(f'  {lbl:>8}  {r2:>8.3f}  {mae:>8.3f}  {mask.sum():>10}')

  Test R²  : 0.9320   (target ≥ 0.90)
  Test MAE : 0.266 ton/ha

R² theo Yield Range:
     Range        R²       MAE   N samples
  --------------------------------------------
       0-2     0.927     0.082        1607
       2-5     0.925     0.115        1486
      5-10     0.862     0.292         553
     10-20     0.851     0.708         253
       20+     0.528     4.195         100


In [ ]:
# ── Cell 9: In ra X_train, X_test, y_train, y_test ───────────────────────────
print('━' * 55)
print('X_train')
print('━' * 55)
print(X_train)

print('\n' + '━' * 55)
print('X_test')
print('━' * 55)
print(X_test)

print('\n' + '━' * 55)
print('y_train  (Yield — BoxCox-transformed)')
print('━' * 55)
print(y_train)

print('\n' + '━' * 55)
print('y_test   (Yield — ton/ha, raw)')
print('━' * 55)
print(y_test)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
X_train
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
              Area  District  Season  Avg Temp  Avg Humidity  Crop Name  \
0       181.365458        58       0      25.0          72.5         36   
1       442.971087        38       1      16.0          85.0         60   
2       734.868770        10       2      20.0          75.0         40   
3       248.532413        60       2      25.5          87.5         37   
4         5.590235        15       1      28.0          48.8          2   
...            ...       ...     ...       ...           ...        ...   
15995   246.378788        31       1      26.5          75.0         17   
15996   764.978583        53       0      11.5          77.5         51   
15997  3294.836052        62       0      29.0          67.5         30   
15998   144.856832         9       1      15.0          90.0         43   
15999   371.961726        46       1      26.5         

: 